<a href="https://colab.research.google.com/github/zeynepdanis/Neural-Network/blob/main/%C4%B0ki_Gizli_Katmanl%C4%B1_MLPNN_%E2%80%94_Bias_Dahil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [4]:
#  Aktivasyon fonksiyonu ve türevi
def sigmoid(x):
    """s(x) = 1 / (1 + e^-x)"""
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_turev(s_x):
    """s'(x) = s(x) * (1 - s(x))   [s_x = sigmoid çıktısı]"""
    return s_x * (1.0 - s_x)


In [17]:
class MLPNN:
    def __init__(self, n_giris, n_gizli1, n_gizli2, n_cikis,
                 ogrenme_katsayisi=0.1, tohum=42):
        np.random.seed(tohum)
        self.alpha = ogrenme_katsayisi
        self.W  = np.random.randn(n_gizli1, n_giris)  * 0.1
        self.V  = np.random.randn(n_gizli2, n_gizli1) * 0.1
        self.U  = np.random.randn(n_cikis,  n_gizli2) * 0.1
        self.b1 = np.zeros(n_gizli1)
        self.b2 = np.zeros(n_gizli2)
        self.b3 = np.zeros(n_cikis)

    def ileri_besleme(self, X):
        F_net = X @ self.W.T + self.b1
        F_out = sigmoid(F_net)
        H_net = F_out @ self.V.T + self.b2
        H_out = sigmoid(H_net)
        Y_net = H_out @ self.U.T + self.b3
        Y_out = sigmoid(Y_net)
        return F_net, F_out, H_net, H_out, Y_net, Y_out

    def geriye_yayilim(self, X, Y_gercek, F_net, F_out,
                       H_net, H_out, Y_net, Y_out):
        E4 = (Y_gercek - Y_out) * sigmoid_turev(Y_out)
        E3 = (E4 @ self.U) * sigmoid_turev(H_out)
        E2 = (E3 @ self.V) * sigmoid_turev(F_out)
        self.U  += self.alpha * (E4.T @ H_out)
        self.b3 += self.alpha * E4.sum(axis=0)
        self.V  += self.alpha * (E3.T @ F_out)
        self.b2 += self.alpha * E3.sum(axis=0)
        self.W  += self.alpha * (E2.T @ X)
        self.b1 += self.alpha * E2.sum(axis=0)

    def egit(self, X, Y, epoch=1000, verbose=True, verbose_adim=100):
        kayip_gecmisi = []
        for ep in range(1, epoch + 1):
            F_net, F_out, H_net, H_out, Y_net, Y_out = self.ileri_besleme(X)
            mse = np.mean((Y - Y_out) ** 2)
            kayip_gecmisi.append(mse)
            self.geriye_yayilim(X, Y, F_net, F_out, H_net, H_out, Y_net, Y_out)
            if verbose and ep % verbose_adim == 0:
                print(f"Epoch {ep:6d} / {epoch}  |  MSE = {mse:.6f}")
        return kayip_gecmisi

    def tahmin_et(self, X):
        _, _, _, _, _, Y_out = self.ileri_besleme(X)
        return Y_out


In [18]:

#  Veri Seti — 16 örnek, 4 giriş → 2 çıkış

X = np.array([
    [0, 0, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0],
    [0, 0, 1, 1],
    [0, 1, 0, 0],
    [0, 1, 0, 1],
    [0, 1, 1, 0],
    [0, 1, 1, 1],
    [1, 0, 0, 0],
    [1, 0, 0, 1],
    [1, 0, 1, 0],
    [1, 0, 1, 1],
    [1, 1, 0, 0],
    [1, 1, 0, 1],
    [1, 1, 1, 0],
    [1, 1, 1, 1],
], dtype=float)

Y = np.array([
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [1, 0],
    [1, 0],
    [1, 0],
    [1, 0],
    [1, 0],
    [1, 0],
    [1, 0],
    [1, 0],
], dtype=float)

In [29]:

#  Ağ Parametreleri
N_GIRIS   = 4
N_GIZLI1  = 8
N_GIZLI2  = 6
N_CIKIS   = 2
ALPHA     = 0.5
EPOCH     = 10000

In [30]:

model = MLPNN(
    n_giris=N_GIRIS,
    n_gizli1=N_GIZLI1,
    n_gizli2=N_GIZLI2,
    n_cikis=N_CIKIS,
    ogrenme_katsayisi=ALPHA,
    tohum=42
)

print("  İki Gizli Katmanlı — Eğitim")
print(f"  Mimari : {N_GIRIS} → {N_GIZLI1} → {N_GIZLI2} → {N_CIKIS}")
print(f"  α = {ALPHA},  Epoch = {EPOCH}")

kayip = model.egit(X, Y, epoch=EPOCH, verbose=True, verbose_adim=1000)

  İki Gizli Katmanlı — Eğitim
  Mimari : 4 → 8 → 6 → 2
  α = 0.5,  Epoch = 10000
Epoch   1000 / 10000  |  MSE = 0.000065
Epoch   2000 / 10000  |  MSE = 0.000028
Epoch   3000 / 10000  |  MSE = 0.000017
Epoch   4000 / 10000  |  MSE = 0.000013
Epoch   5000 / 10000  |  MSE = 0.000010
Epoch   6000 / 10000  |  MSE = 0.000008
Epoch   7000 / 10000  |  MSE = 0.000007
Epoch   8000 / 10000  |  MSE = 0.000006
Epoch   9000 / 10000  |  MSE = 0.000005
Epoch  10000 / 10000  |  MSE = 0.000005


In [31]:

#  Tahmin Sonuçları
tahminler     = model.tahmin_et(X)
tahminler_bin = (tahminler >= 0.5).astype(int)

print("\n── Tahmin Tablosu ──")
print(f"{'#':>2}  {'x1 x2 x3 x4':12}  {'Hedef':8}  {'Tahmin (sürekli)':22}  {'Yuvarlak':10}  Sonuç")
print("-" * 72)
for i in range(len(X)):
    grs = "  ".join(map(str, X[i].astype(int)))
    hdf = str(Y[i].astype(int).tolist())
    sre = f"[{tahminler[i,0]:.4f}, {tahminler[i,1]:.4f}]"
    yur = str(tahminler_bin[i].tolist())
    ok  = "✓" if (Y[i] == tahminler_bin[i]).all() else "✗"
    print(f"{i:>2}  {grs:12}  {hdf:8}  {sre:22}  {yur:10}  {ok}")

genel_acc = (Y == tahminler_bin).all(axis=1).mean()
print(f"\nGenel Doğruluk : {genel_acc*100:.1f}%  ({int(genel_acc*16)}/16 örnek)")
print(f"Son MSE        : {kayip[-1]:.8f}")


── Tahmin Tablosu ──
 #  x1 x2 x3 x4   Hedef     Tahmin (sürekli)        Yuvarlak    Sonuç
------------------------------------------------------------------------
 0  0  0  0  0    [0, 0]    [0.0033, 0.0011]        [0, 0]      ✓
 1  0  0  0  1    [0, 0]    [0.0027, 0.0011]        [0, 0]      ✓
 2  0  0  1  0    [0, 0]    [0.0027, 0.0011]        [0, 0]      ✓
 3  0  0  1  1    [0, 0]    [0.0023, 0.0011]        [0, 0]      ✓
 4  0  1  0  0    [0, 0]    [0.0027, 0.0011]        [0, 0]      ✓
 5  0  1  0  1    [0, 0]    [0.0023, 0.0011]        [0, 0]      ✓
 6  0  1  1  0    [0, 0]    [0.0022, 0.0011]        [0, 0]      ✓
 7  0  1  1  1    [0, 0]    [0.0020, 0.0011]        [0, 0]      ✓
 8  1  0  0  0    [1, 0]    [0.9976, 0.0019]        [1, 0]      ✓
 9  1  0  0  1    [1, 0]    [0.9974, 0.0019]        [1, 0]      ✓
10  1  0  1  0    [1, 0]    [0.9974, 0.0019]        [1, 0]      ✓
11  1  0  1  1    [1, 0]    [0.9971, 0.0018]        [1, 0]      ✓
12  1  1  0  0    [1, 0]    [0.9974, 0.0019

In [32]:

#  Hata Matrisi ve Performans Metrikleri

def hesapla_metrikler(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    toplam = tp + tn + fp + fn

    dogruluk = (tp + tn) / toplam if toplam > 0 else np.nan
    kesinlik = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    duyarlilik = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1 = (
        2 * kesinlik * duyarlilik / (kesinlik + duyarlilik)
        if (kesinlik + duyarlilik) > 0
        else np.nan
    )

    return {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Doğruluk": dogruluk,
        "Kesinlik": kesinlik,
        "Duyarlılık": duyarlilik,
        "F1 Skoru": f1
    }


def format_deger(v):
    if isinstance(v, float):
        return "—" if np.isnan(v) else f"{v:.4f}"
    return v


def hata_matrisi_yazdir(ad, m):
    print(f"\n╔════════════════════════════════════╗")
    print(f"║        Hata Matrisi: {ad:<12} ║")
    print(f"╚════════════════════════════════════╝")
    print()
    print("                 Tahmin")
    print("              0          1")
    print("        ┌──────────┬──────────┐")
    print(f"Gerçek 0│ {m['TN']:^8} │ {m['FP']:^8} │")
    print("        ├──────────┼──────────┤")
    print(f"Gerçek 1│ {m['FN']:^8} │ {m['TP']:^8} │")
    print("        └──────────┴──────────┘")


# ─────────────────────────────────────────────
# Metrikleri Hesapla
# ─────────────────────────────────────────────

m1 = hesapla_metrikler(Y[:, 0], tahminler_bin[:, 0])
m2 = hesapla_metrikler(Y[:, 1], tahminler_bin[:, 1])


# ─────────────────────────────────────────────
# Hata Matrislerini Yazdır
# ─────────────────────────────────────────────

print("\n" + "═" * 50)
print("HATA MATRİSLERİ")
print("═" * 50)

hata_matrisi_yazdir("y1", m1)
hata_matrisi_yazdir("y2", m2)


# ─────────────────────────────────────────────
# Performans Tablosu
# ─────────────────────────────────────────────

performans_df = pd.DataFrame({
    "Metrik": [
        "TP",
        "TN",
        "FP",
        "FN",
        "Doğruluk",
        "Kesinlik",
        "Duyarlılık",
        "F1 Skoru"
    ],
    "y1": [
        m1["TP"],
        m1["TN"],
        m1["FP"],
        m1["FN"],
        m1["Doğruluk"],
        m1["Kesinlik"],
        m1["Duyarlılık"],
        m1["F1 Skoru"]
    ],
    "y2": [
        m2["TP"],
        m2["TN"],
        m2["FP"],
        m2["FN"],
        m2["Doğruluk"],
        m2["Kesinlik"],
        m2["Duyarlılık"],
        m2["F1 Skoru"]
    ]
})

performans_df["y1"] = performans_df["y1"].apply(format_deger)
performans_df["y2"] = performans_df["y2"].apply(format_deger)

print("\n" + "═" * 50)
print("PERFORMANS ÖLÇEKLERİ")
print("═" * 50)
print(performans_df.to_string(index=False))


══════════════════════════════════════════════════
HATA MATRİSLERİ
══════════════════════════════════════════════════

╔════════════════════════════════════╗
║        Hata Matrisi: y1           ║
╚════════════════════════════════════╝

                 Tahmin
              0          1
        ┌──────────┬──────────┐
Gerçek 0│    8     │    0     │
        ├──────────┼──────────┤
Gerçek 1│    0     │    8     │
        └──────────┴──────────┘

╔════════════════════════════════════╗
║        Hata Matrisi: y2           ║
╚════════════════════════════════════╝

                 Tahmin
              0          1
        ┌──────────┬──────────┐
Gerçek 0│    16    │    0     │
        ├──────────┼──────────┤
Gerçek 1│    0     │    0     │
        └──────────┴──────────┘

══════════════════════════════════════════════════
PERFORMANS ÖLÇEKLERİ
══════════════════════════════════════════════════
    Metrik     y1      y2
        TP 8.0000  0.0000
        TN 8.0000 16.0000
        FP 0.0000  0.